# nb_03 — Color-magnitude diagram

Goal (see `docs/SPEC_V01.md`, rough plan step 3): color-magnitude diagram (CMD), `g-r` vs `r`
by default — the color and magnitude bands are plain variables below, change them as you like.
Works on `dia_object_lc_hq` — nb_01's base HQ sample with nb_02's LC-stat columns already
merged into it in place.

**Resolves the spec's magnitude question.** `diaObject` itself has no usable per-band static
magnitude for a CMD: `{band}_psfFluxMean` is a difference-image statistic (near zero for a
non-varying source, useless as a brightness), and its science-image counterpart,
`{band}_scienceFluxMean`, turned out to exist but be too sparse to rely on (only ~20% of
objects had it populated, checked against nb-v01's smaller subset). Going through the
`Object` (coadd) table instead would need a spatial crossmatch — slow, and ambiguous in
crowded fields — so that's ruled out.

The right table is forced photometry on the science image at each diaObject's position, at
every visit — `ForcedSourceOnDiaObject`. It *is* reachable through LSDB: it's the nested
`diaObjectForcedSource` column already present in `dia_object_lc_hq`. Each row's `psfFlux`/
`psfMag` there is the calibrated science-image measurement, not a diff (`psfDiffFlux` is the
actual diff-image value, alongside it, for comparison). The median `psfMag` per band across
the forced-photometry rows covered ~99.7% of objects on nb-v01's subset — section 1 computes
the same thing at HQ scale.

**At HQ scale, the CMD/color-color plots below use `hexbin` (2D histograms) instead of
per-point scatter** — nb-v01's alpha-blended scatter worked at ~7,000 points but saturates
into a solid blob well before HQ's ~399k, and there's no more `field` column to split panels
by anyway (the point of this refactor is to treat the sample as a whole).

In [ ]:
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lsdb
from datapaths import Datapaths

sys.path.insert(0, str(Path.cwd().parent / "src"))
from dataio import select_slice

from dask.distributed import Client

client = Client(n_workers=4, threads_per_worker=1, memory_limit="auto")

dp = Datapaths()
hq_path = dp["dp2_subset"] / "dia_object_lc_hq"
hq_cat = lsdb.open_catalog(hq_path)
print(hq_cat.npartitions, "partitions")
hq_cat

## 0. Pick a slice of the HQ sample to explore

Same pattern as nb_02: `select_slice` (`src/dataio/hq_sample.py`) gets you either one
partition or a cone search's worth of `hq_cat` to look at directly, without touching the
whole ~399k-object sample the CMD below is computed over.

In [ ]:
SLICE_MODE = "partition"  # "partition" or "cone_search"
PARTITION_INDEX = 200
CONE_RA, CONE_DEC, CONE_RADIUS_ARCSEC = 150.0, 2.0, 1800  # ~0.5 deg

slice_cat = select_slice(
    hq_cat,
    mode=SLICE_MODE,
    partition_index=PARTITION_INDEX,
    ra=CONE_RA,
    dec=CONE_DEC,
    radius_arcsec=CONE_RADIUS_ARCSEC,
)
slice_df = slice_cat.compute()
print(f"{SLICE_MODE}: {len(slice_df)} objects")
slice_df.drop(columns=["diaSource", "diaObjectForcedSource"]).head()

## 1. Per-band magnitude via forced photometry (`diaObjectForcedSource`)

Same `map_partitions` pattern as nb_02, but this time on `diaObjectForcedSource` (nested,
per-visit forced photometry) instead of `diaSource` (nested, per-visit *detections* — biased
toward brighter/outburst epochs, since a detection has to clear a threshold). `func` gets a
whole partition (`nested_pandas.NestedFrame`) and writes to a temporary path with
`resume=True`, then promotes it into `dia_object_lc_hq` in place — same reasoning as nb_02's
section 2 (plain `resume=True` straight onto `dia_object_lc_hq` would skip every pixel that's
already there, silently not adding this section's columns to any of them).

**Vectorized the same way as nb_02**: `.explode("diaObjectForcedSource")` to a long,
one-row-per-visit table, then a `groupby` per band instead of looping `df.iloc[i]` per object.
Checked against the original loop version on a real 75-partition, 2,713-object slice:
identical results (to floating-point precision — median differs by ~1e-6, a float32-storage
rounding artifact, not a logic difference) at **~50x** the speed (2.1s vs 107s).

Median, not a flux-error-weighted mean: keeps the per-band aggregation consistent with nb_02's
robust-statistics style, at the cost of not being the statistically optimal combination — worth
revisiting if a workshop attendee needs tighter photometry than a demo CMD does.

Two quality cuts added on top of the plain median (missing from the first version of this
notebook — see section 3 for why extendedness isn't among them):
- **`MIN_FORCED_PER_BAND`**: a band's median is only kept if at least this many unflagged
  forced points went into it, `NaN` otherwise (an early version kept any band with `>0`
  points, which let a handful of noisy 1-2-point medians through and produced implausible
  `g-r` outliers outside ±2).
- **`FORCED_FLAG_COLS`**: excludes forced-source rows carrying any of `diaObjectForcedSource`'s
  own pixel/PSF-fit quality flags (saturated, cosmic ray, edge, interpolated, no-data pixels;
  a failed or invalid PSF flux fit) before computing the median.

Also computes `max_reliability` — the maximum of `diaSource.reliability` (DP2's per-detection
real/bogus ML score) across an object's light curve — for the reliability split in section 3.
`max`, not `median`: see that section for why.
</cell id="6541ed0c">

### Heavy-calculation flag

Same idea as nb_02: `map_partitions` below now runs across the whole ~399k-object HQ sample
rather than nb-v01's 7,036-object subset, vectorized per-partition (see section 1's markdown),
writing to a temporary path with `resume=True` and then promoting it into `dia_object_lc_hq`
(a crash partway only loses the temporary path's not-yet-written partitions, never the
existing collection). Extrapolating the measured 2.1s/2,713-object rate to ~399k objects
projects to roughly 5 minutes — not checked against the real full sample yet (only small
slices — see `docs/changelog.md`). Same caveat as nb_02 applies: `resume=True` doesn't know if
`band_mags_partition` changed, only which pixels already exist in the temporary path — delete
it first (or pass `overwrite=True` for that write instead) for a clean rebuild after editing
the function.

`RUN_HEAVY_CALC` gates the `map_partitions`/write/register step; leave it `False` (the
default) to reuse the columns `dia_object_lc_hq` already has instead of recomputing per-band
forced-photometry medians, and go straight to the CMD/color-color sections below.
`band_mags_partition` and its constants are still defined either way, so the logic is there to
read even when it doesn't run. **First time populating this branch's registry:** set it `True`
once so `dia_object_lc_hq` actually has these columns before nb_04 tries to read them.

In [6]:
RUN_HEAVY_CALC = True

In [ ]:
BANDS = "ugrizy"
MIN_FORCED_PER_BAND = 5

# Photometric quality flags on diaObjectForcedSource — exclude rows affected by any of these
# before taking the per-band median. Not exhaustive (diaSource has its own, separate per-epoch
# flags used by nb_02's stats), but these are the ones bearing directly on this section's
# forced-photometry magnitudes.
FORCED_FLAG_COLS = [
    "pixelFlags_bad", "pixelFlags_cr", "pixelFlags_crCenter", "pixelFlags_edge",
    "pixelFlags_interpolated", "pixelFlags_interpolatedCenter", "pixelFlags_nodata",
    "pixelFlags_saturated", "pixelFlags_saturatedCenter", "pixelFlags_suspect",
    "pixelFlags_suspectCenter", "psfFlux_flag", "invalidPsfFlag",
]


def band_mags_partition(df):
    ids = df["diaObjectId"].to_numpy()

    fs_cols = ["diaObjectId", "band", "psfMag"] + FORCED_FLAG_COLS
    fs = df[["diaObjectId", "diaObjectForcedSource"]].explode("diaObjectForcedSource")
    fs = fs[fs_cols]

    bad = fs[FORCED_FLAG_COLS].any(axis=1)
    good = fs[~bad & np.isfinite(fs["psfMag"])]

    df = df.copy()
    for b in BANDS:
        grp = good.loc[good["band"] == b].groupby("diaObjectId")["psfMag"]
        counts = grp.size()
        med = grp.median().where(counts >= MIN_FORCED_PER_BAND)
        df[f"{b}_mag_median"] = med.reindex(ids).to_numpy()

    # DP2's per-detection real/bogus score, from the diaSource stream (not
    # diaObjectForcedSource) — see the markdown below for why this is `max`, not `median`, and
    # why it's not a 0.9-style real/bogus cut.
    ds = df[["diaObjectId", "diaSource"]].explode("diaSource")[["diaObjectId", "reliability"]]
    max_reliability = ds.groupby("diaObjectId")["reliability"].max()
    df["max_reliability"] = max_reliability.reindex(ids).to_numpy()

    return df


if RUN_HEAVY_CALC:
    tmp_path = hq_path.parent / (hq_path.name + "_nb03_tmp")

    with_mags_cat = hq_cat.map_partitions(band_mags_partition)
    with_mags_cat.write_catalog(tmp_path, resume=True, create_thumbnail=False, progress_bar=False)

    if hq_path.exists():
        shutil.rmtree(hq_path)
    tmp_path.rename(hq_path)
    print("promoted", tmp_path, "->", hq_path)

    dp.register(
        name="dia_object_lc_hq",
        type="dp2_subset",
        fmt="hats-collection",
        src_path=str(hq_path / "collection.properties"),
        tags=["nb01", "nb02", "nb03", "dp2", "diaObject", "cmd", "forced-photometry", "hq"],
        notes=(
            "nb_01's base HQ sample plus nb_02's LC stats, plus {band}_mag_median (per band, "
            "median diaObjectForcedSource.psfMag, quality-flagged rows excluded, "
            f">={MIN_FORCED_PER_BAND} points required) and max_reliability (max "
            "diaSource.reliability) columns, from nb_03."
        ),
        overwrite_history=True,
    )
    dp.print_paths(name="dia_object_lc_hq")
else:
    print(f"RUN_HEAVY_CALC=False — skipping write/register; {hq_path} already has these columns.")
client.close()

## 2. Color-magnitude diagram

Reopen the written collection to confirm the new columns persisted, then pull what's needed
for the CMD into a flat dataframe. `COLOR_BAND_BLUE` / `COLOR_BAND_RED` / `MAG_BAND` are plain
variables — change them for a different color or magnitude band.


In [ ]:
hq_cat = lsdb.open_catalog(hq_path)
print(sorted(hq_cat.columns))

COLOR_BAND_BLUE = "g"
COLOR_BAND_RED = "r"
MAG_BAND = "r"

mag_cols = sorted({f"{COLOR_BAND_BLUE}_mag_median", f"{COLOR_BAND_RED}_mag_median", f"{MAG_BAND}_mag_median"})
extra_cols = ["i_mag_median", "max_reliability"]  # not required for the plain CMD, kept for later sections

cmd_df = hq_cat[["diaObjectId", "r_amp_p90p10", *mag_cols, *extra_cols]].compute()
cmd_df = cmd_df.dropna(subset=mag_cols)
cmd_df["color"] = cmd_df[f"{COLOR_BAND_BLUE}_mag_median"] - cmd_df[f"{COLOR_BAND_RED}_mag_median"]
print(cmd_df.shape)
cmd_df[["color", f"{MAG_BAND}_mag_median"]].describe()

### Plots

At HQ scale (hundreds of thousands of points after the quality cuts above), per-point scatter
saturates into a solid blob, so these use `hexbin` (2D histograms) instead — one panel for
raw point density, one for median r-band amplitude per bin (does variability track a
particular part of the CMD?).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)

hb0 = axes[0].hexbin(cmd_df["color"], cmd_df[f"{MAG_BAND}_mag_median"], gridsize=80, cmap="viridis", mincnt=1)
axes[0].set_title(f"point density (n={len(cmd_df):,})")
fig.colorbar(hb0, ax=axes[0], label="objects/bin")

hb1 = axes[1].hexbin(
    cmd_df["color"], cmd_df[f"{MAG_BAND}_mag_median"], C=cmd_df["r_amp_p90p10"],
    reduce_C_function=np.median, gridsize=80, cmap="plasma", mincnt=1,
)
axes[1].set_title("median r-band P90-P10 amplitude")
fig.colorbar(hb1, ax=axes[1], label="amplitude [mag]")

for ax in axes:
    ax.set_xlabel(f"{COLOR_BAND_BLUE}-{COLOR_BAND_RED}")
axes[0].set_ylabel(f"{MAG_BAND} (median forced-phot mag)")
axes[0].invert_yaxis()

fig.tight_layout()

In [ ]:
amp_bins = [(0, 0.2, "A < 0.2"), (0.2, 0.4, "0.2 ≤ A < 0.4"), (0.4, np.inf, "A ≥ 0.4")]

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharex=True, sharey=True)

for ax, (lo, hi, label) in zip(axes, amp_bins):
    sub = cmd_df[(cmd_df["r_amp_p90p10"] >= lo) & (cmd_df["r_amp_p90p10"] < hi)]
    ax.hexbin(sub["color"], sub[f"{MAG_BAND}_mag_median"], gridsize=50, cmap="viridis", mincnt=1)
    ax.set_title(f"{label} (n={len(sub):,})")
    ax.set_xlabel(f"{COLOR_BAND_BLUE}-{COLOR_BAND_RED}")

axes[0].invert_yaxis()
axes[0].set_ylabel(f"{MAG_BAND} (median forced-phot mag)")

fig.tight_layout()

## 3. Quality flags: reliability, and the extendedness gap

The natural point-source/extended split — `extendedness`, typically from the coadd `Object`
table — isn't available here. It's not in `diaObject`'s flat columns, nor in the nested
`diaSource` or `diaObjectForcedSource` schemas of this catalog (checked directly against the
HATS schema, not just what happened to already be loaded). DP2's `diaSource` schema does
define an `extendedness` field, so this looks like something dropped when this particular HATS
catalog was built — not something that doesn't exist in DP2 at all. **Doing proper quality
filtering — extendedness among it — is going to need a data path other than this LSDB HATS
collection** (direct Butler/PPDB access, or a HATS catalog built with more columns retained);
this notebook can't do it as-is.

`diaSource.reliability` (DP2's per-detection real/bogus ML score, 0-1) is available, but it's
heavily skewed toward zero for this subset: the 99th percentile across all ~134k individual
detections is only ~0.09, and only ~1.2% of objects ever reach `reliability > 0.5` even at
their *best* epoch (`max`, computed in section 1 — `median` is even more degenerate, since most
epochs sit near zero for essentially every object). So it isn't usable as an absolute "> 0.9 =
real" cut the way it might be on a more mature classifier. Below it's used as a relative split
instead — top 10% of objects by `max_reliability` vs. the rest — which says more about
"unusually high-scoring within this specific sample" than "point source vs. galaxy" or "real
vs. bogus" in any calibrated sense.


In [ ]:
rel_threshold = np.nanpercentile(cmd_df["max_reliability"], 90)
print(f"top-10% max_reliability threshold: {rel_threshold:.4f}")

rel_bins = [(0, rel_threshold, "bottom 90%"), (rel_threshold, np.inf, "top 10%")]

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5), sharex=True, sharey=True)

for ax, (lo, hi, label) in zip(axes, rel_bins):
    sub = cmd_df[(cmd_df["max_reliability"] >= lo) & (cmd_df["max_reliability"] < hi)]
    ax.hexbin(sub["color"], sub[f"{MAG_BAND}_mag_median"], gridsize=50, cmap="viridis", mincnt=1)
    ax.set_title(f"max_reliability: {label} (n={len(sub):,})")
    ax.set_xlabel(f"{COLOR_BAND_BLUE}-{COLOR_BAND_RED}")

axes[0].invert_yaxis()
axes[0].set_ylabel(f"{MAG_BAND} (median forced-phot mag)")

fig.tight_layout()

## 4. Color-color diagram

`g-r` vs `r-i` by default (`COLOR2_BLUE`/`COLOR2_RED` — change for a different second color).
Needs `i_mag_median` too, on top of `g_mag_median`/`r_mag_median`, so this drops a few more
objects to `NaN`s than the plain CMD above.


In [ ]:
COLOR2_BLUE = "r"
COLOR2_RED = "i"

cc_cols = sorted({f"{COLOR_BAND_BLUE}_mag_median", f"{COLOR_BAND_RED}_mag_median", f"{COLOR2_BLUE}_mag_median", f"{COLOR2_RED}_mag_median"})
cc_df = hq_cat[["diaObjectId", *cc_cols]].compute()
cc_df = cc_df.dropna(subset=cc_cols)
cc_df["color1"] = cc_df[f"{COLOR_BAND_BLUE}_mag_median"] - cc_df[f"{COLOR_BAND_RED}_mag_median"]
cc_df["color2"] = cc_df[f"{COLOR2_BLUE}_mag_median"] - cc_df[f"{COLOR2_RED}_mag_median"]
print(cc_df.shape)

fig, ax = plt.subplots(figsize=(6, 5.5))
hb = ax.hexbin(cc_df["color1"], cc_df["color2"], gridsize=70, cmap="viridis", mincnt=1)
ax.set_xlabel(f"{COLOR_BAND_BLUE}-{COLOR_BAND_RED}")
ax.set_ylabel(f"{COLOR2_BLUE}-{COLOR2_RED}")
ax.set_title(f"n={len(cc_df):,}")
fig.colorbar(hb, ax=ax, label="objects/bin")
fig.tight_layout()

## Next

`dia_object_lc_hq` now carries the light curves, nb_02's stats, and the new per-band median
magnitudes (plus `max_reliability`) together (once `RUN_HEAVY_CALC=True` has been run at least
once on this branch).

Open questions carried forward, not resolved here:
- **Aggregation choice.** Median `psfMag` per band is simple and robust but not statistically
  optimal — a `psfFluxErr`-weighted mean would use the per-visit uncertainties properly.
- **Real quality filtering needs a different data path.** `extendedness` and any deeper
  point-source/extended or star/galaxy separation aren't reachable through this LSDB HATS
  collection (see section 3) — would need direct Butler/PPDB access, or a HATS catalog rebuilt
  with more columns retained.
- **`reliability`'s calibration.** DP2's `diaSource.reliability` was heavily skewed near zero
  in nb-v01's subset (section 3); the top-10% split used here is relative, not an absolute
  real/bogus cut — worth revisiting once/if a better-calibrated version is available, and
  re-checking whether the skew holds at HQ scale.
- **`hexbin`'s `gridsize` values (50-80) are eyeballed, not tuned** — worth revisiting once
  the actual HQ-scale point counts and dynamic range are known.